# The arXiv corpus, end to end

**One notebook from the raw arXiv sources to a corpus `tessera build` can consume**, with two
clusterings over it and a label set over each. It exists so the demo corpus is *reproducible* —
the pipeline that made it is readable, re-runnable and versioned, rather than a sequence of
one-off scripts whose output has to be archived because nobody can make it again.

## What it produces

A directory of build inputs and the configuration that reads them:

| File | What it is |
|---|---|
| `points.parquet` | one row per paper: identity, UMAP position, and the attribute columns a client can filter and draw on |
| `pairs.parquet` | the `(entity, term)` access relation — **the access control**, derived from arXiv's own categories |
| `terms.parquet` | `term_id → descriptor`, so a grant can be written in category names rather than integers |
| `artifacts.parquet` | the clusters and the labels, including each cluster's **parent** where it has one |
| `members.parquet` | which papers belong to which cluster, and which a label was generated from |
| `layers.toml` | the four layers: two clusterings, two label sets |
| `schema.toml` | the attribute columns and their homes |
| `archive.parquet`, `primary_category.parquet` | the two category vocabularies |
| `manifest.json` | what this run did — sizes, parameters, timings |

## The two clusterings are deliberately different shapes

**K-means is flat.** Every cluster is a peer of every other; there are no edges. It is the
control, and it is what most of this system was built and measured against.

**HDBSCAN is a tree**, and its shape is the reason it is here. Its condensed tree gives a genuine
parent/child hierarchy, and — crucially — **a cluster's children do not exhaust it**: a fifth to a
quarter of a parent's points fall out as noise at each split rather than joining any child. That
property is not something a synthetic tree produces by accident, and any construction that assumes
a covering hierarchy is wrong in a way a planted balanced tree will never reveal.

## What this is not

Python is a **consumer** here and never a component. This writes files; nothing in it sits on a
request path, produces a served artifact, or belongs to any trusted computing base.

## 0. Configuration

Everything that decides what this run does is here. The defaults are chosen so the whole notebook
runs end to end in a few minutes; the notes say what to change and what it costs.

In [ ]:
import json, os, time, pathlib, sys
import numpy as np

T0 = time.time()
STEPS = {}          # step name -> seconds, reported at the end and written to the manifest

def step(name):
    """Time a block and record it, so `manifest.json` says what the run actually cost."""
    class _Step:
        def __enter__(self):
            self.t = time.time(); print(f"[{name}] ...", flush=True); return self
        def __exit__(self, *a):
            STEPS[name] = round(time.time() - self.t, 2)
            print(f"[{name}] {STEPS[name]}s", flush=True)
    return _Step()

# --- where the sources are, and where the output goes ------------------------------------------
# TESSERA_DATA is the checkout holding `data/`. A worktree has no `data/` of its own.
DATA = pathlib.Path(os.environ.get("TESSERA_DATA", "/home/joe/code/tessera/data"))
OUT  = pathlib.Path(os.environ.get("TESSERA_NOTEBOOK_OUT", DATA / "notebook"))
OUT.mkdir(parents=True, exist_ok=True)

# --- how much of the corpus ---------------------------------------------------------------------
# `None` takes all 2,422,486 papers. The sample is drawn *uniformly*, which keeps the category
# skew and the density structure the clusterings depend on; taking a prefix would instead take the
# oldest papers and change both.
SAMPLE = int(os.environ.get("TESSERA_NOTEBOOK_SAMPLE", 200_000)) or None

SEED = 0

# --- geometry -------------------------------------------------------------------------------------
# **UMAP is reused by default, not recomputed.** `data/geometry.parquet` holds the 2D embedding of
# the whole corpus, and it is a *hashed artifact* rather than a reproducible one: it was built with
# cuML on a GPU, which is not bit-reproducible even under a fixed seed. Reusing it is what makes
# this notebook's output comparable with every measurement already taken against the corpus.
#
# Set RECOMPUTE_UMAP = True to run the embedding here instead: PCA to 64 components followed by CPU
# UMAP, which *is* reproducible under a fixed random_state (umap-learn drops to a single thread
# when given one). It costs roughly a minute per 10k points and wants the sample kept small.
RECOMPUTE_UMAP = os.environ.get("TESSERA_NOTEBOOK_UMAP", "reuse") == "recompute"
PCA_DIM = 64
UMAP_PARAMS = dict(n_neighbors=15, min_dist=0.1, n_components=2, random_state=SEED)

# --- the two clusterings --------------------------------------------------------------------------
KMEANS_K = 64

# `min_cluster_size` is the knob that decides how deep and how many. It scales with the sample so
# the shape of the tree is comparable across sizes rather than dissolving into noise at the small
# end and into a handful of giants at the large one.
def _mcs(n):
    return max(50, n // 400)

# --- labels ------------------------------------------------------------------------------------
LABEL_TERMS = 3          # how many TF-IDF terms make up one label
LABEL_SAMPLE = 200       # documents a label is generated from — its generating set

print(f"sources  {DATA}")
print(f"output   {OUT}")
print(f"sample   {SAMPLE if SAMPLE else 'the whole corpus'}")
print(f"umap     {'recomputed here' if RECOMPUTE_UMAP else 'reused from geometry.parquet'}")

## 1. Load the corpus and join the sources

Two real sources, already joined into `corpus.parquet` by `probes/build_corpus.py`: the arXiv
metadata snapshot, and one BGE topic embedding per paper. The join was verified clean — all
2,422,486 embedding IDs match a metadata `id` verbatim.

What this cell adds is the **prose** (title and abstract, which the labels are generated from) and
the **geometry**, then reduces all three to the sample.

In [ ]:
import pyarrow as pa, pyarrow.parquet as pq

with step("load metadata"):
    corpus = pq.read_table(DATA / "corpus.parquet",
                           columns=["entity_id", "id", "categories", "v1_created"])
    n_full = corpus.num_rows

with step("load prose"):
    prose = pq.read_table(DATA / "demo" / "prose.parquet", columns=["entity_id", "title", "abstract"])

# Both tables are in `entity_id` order over the same dense 0..n space, which `build_corpus.py`
# assigns in `(v1_created, id)` order. Asserted rather than assumed: a silent misalignment here
# would attach every paper's title to a different paper's position.
assert prose.num_rows == n_full
assert np.array_equal(corpus.column("entity_id").to_numpy(), prose.column("entity_id").to_numpy()), \
    "corpus and prose disagree about entity order"

print(f"{n_full:,} papers")

### The sample

Drawn uniformly and then **sorted**, so the output is in a stable order whatever the sample size.
Entities are renumbered densely from zero: the notebook's corpus is self-contained, and the
original arXiv identifier travels in its own column so a row can always be traced back.

In [ ]:
with step("sample"):
    rng = np.random.default_rng(SEED)
    if SAMPLE is None or SAMPLE >= n_full:
        take = np.arange(n_full)
    else:
        take = np.sort(rng.choice(n_full, SAMPLE, replace=False))
    n = len(take)

    arxiv_id  = np.asarray(corpus.column("id"))[take]
    categories = np.asarray(corpus.column("categories"))[take]
    created   = corpus.column("v1_created").to_numpy(zero_copy_only=False)[take]
    title     = np.asarray(prose.column("title"))[take]
    abstract  = np.asarray(prose.column("abstract"))[take]

print(f"{n:,} papers sampled from {n_full:,}")

## 2. UMAP

Reused by default (see the configuration note); recomputed here when asked. The recompute path is
PCA to 64 components followed by UMAP to 2, which is the same shape as
`probes/build_geometry.py` — the difference is CPU rather than GPU, and therefore reproducible.

In [ ]:
if not RECOMPUTE_UMAP:
    with step("umap (reused)"):
        geo = pq.read_table(DATA / "geometry.parquet", columns=["entity_id", "x", "y"])
        assert geo.num_rows == n_full
        gx, gy = geo.column("x").to_numpy(), geo.column("y").to_numpy()
        xy = np.column_stack([gx[take], gy[take]]).astype(np.float32)
else:
    with step("load embeddings"):
        # Streamed and filtered to the sample, so memory is the sample's rows rather than the
        # corpus's: 2.42M x 1024 float32 is ten gigabytes.
        ent_of_id = dict(zip(np.asarray(corpus.column("id")).tolist(),
                             corpus.column("entity_id").to_numpy().tolist()))
        wanted = {int(e): i for i, e in enumerate(take)}
        DIM = 1024
        X = np.empty((n, DIM), dtype=np.float32)
        seen = np.zeros(n, dtype=bool)
        pf = pq.ParquetFile(DATA / "arxiv_papers_embeds.parquet")
        for batch in pf.iter_batches(batch_size=65536, columns=["paper_id", "embedding"]):
            ids = batch.column("paper_id").to_pylist()
            rows = [wanted.get(ent_of_id[i]) for i in ids]
            hit = [(k, r) for k, r in enumerate(rows) if r is not None]
            if not hit:
                continue
            emb = np.asarray(batch.column("embedding").flatten(), dtype=np.float32).reshape(len(ids), DIM)
            for k, r in hit:
                X[r] = emb[k]; seen[r] = True
        assert seen.all(), f"{(~seen).sum()} sampled papers have no embedding"
        # BGE is cosine-conventional, so L2-normalise before any Euclidean step.
        X /= np.linalg.norm(X, axis=1, keepdims=True)

    with step("pca"):
        mu = X.mean(axis=0)
        cov = np.cov((X - mu).T.astype(np.float64))
        evals, evecs = np.linalg.eigh(cov)
        P = evecs[:, ::-1][:, :PCA_DIM].astype(np.float32)
        Xp = (X - mu) @ P
        kept = evals[::-1][:PCA_DIM].sum() / evals.sum()
        print(f"{PCA_DIM} components keep {100 * kept:.1f}% of the variance")
        del X

    with step("umap"):
        import umap
        xy = np.asarray(umap.UMAP(**UMAP_PARAMS).fit_transform(Xp), dtype=np.float32)

print(f"geometry {xy.shape}, x {xy[:,0].min():.2f}..{xy[:,0].max():.2f}, "
      f"y {xy[:,1].min():.2f}..{xy[:,1].max():.2f}")

## 3. K-means — the flat clustering

Every cluster a peer of every other, no edges. This is the control: it is the shape the system was
built and measured against, and it is what the tree in the next section has to be compared with.

In [ ]:
from sklearn.cluster import KMeans

with step("k-means"):
    km = KMeans(n_clusters=KMEANS_K, random_state=SEED, n_init="auto").fit(xy)
    kmeans_label = km.labels_.astype(np.int32)

sizes = np.bincount(kmeans_label, minlength=KMEANS_K)
print(f"{KMEANS_K} clusters, sizes {sizes.min():,}..{sizes.max():,} (median {int(np.median(sizes)):,})")

## 4. HDBSCAN — the tree

The `hdbscan` package rather than scikit-learn's implementation, for one reason: it exposes
`condensed_tree_`, which **is** the hierarchy. Reconstructing it from a single-linkage tree would
be reimplementing the algorithm's own output.

### Reading the condensed tree

Its rows are `(parent, child, lambda_val, child_size)`. Nodes numbered below the point count are
individual papers; nodes at or above it are clusters, and the first of them is the root.

- A row whose `child_size > 1` is a **cluster edge** — this is the lineage.
- A row whose `child_size == 1` is a **paper detaching** from a cluster. That paper belongs to that
  cluster and to all of its ancestors, and to none of its children.

The second point is where the non-covering property comes from, and it is worth being explicit
about: a paper that detaches at cluster *C* is a member of *C* that **no child of *C* holds**. Those
papers are why a parent's masked count is not the sum of its children's, and why a rollup that
unions the children and calls the result the parent is wrong on every real hierarchy while passing
on every planted one.

In [ ]:
import hdbscan
from collections import defaultdict

with step("hdbscan"):
    mcs = _mcs(n)
    clusterer = hdbscan.HDBSCAN(min_cluster_size=mcs, min_samples=10, core_dist_n_jobs=os.cpu_count())
    clusterer.fit(xy)
    tree = clusterer.condensed_tree_.to_pandas()

selected_noise = float((clusterer.labels_ == -1).mean())
print(f"min_cluster_size {mcs}, {len(set(clusterer.labels_)) - 1} selected clusters, "
      f"{selected_noise:.1%} of papers in none of them")

# The lineage: cluster -> its parent cluster. The root has none.
cluster_edges = tree[tree.child_size > 1]
parent_of = {int(r.child): int(r.parent) for r in cluster_edges.itertuples()}
root = n                                    # the first cluster node is numbered after the papers
nodes = sorted({root} | set(parent_of))
children_of = defaultdict(list)
for c, p in parent_of.items():
    children_of[p].append(c)

# Where each paper detaches — exactly one cluster each.
leaves = tree[tree.child_size == 1]
detach = np.full(n, -1, dtype=np.int64)
detach[leaves.child.to_numpy().astype(np.int64)] = leaves.parent.to_numpy().astype(np.int64)
assert (detach >= 0).all(), "a paper detaches from no cluster, which the condensed tree cannot produce"

# Members of a cluster: the papers detaching at it, plus everything under its descendants. Computed
# bottom-up so each node is visited once rather than once per ancestor.
own = defaultdict(list)
for paper, at in enumerate(detach):
    own[int(at)].append(paper)

def _depth(c):
    d = 0
    while c in parent_of:
        c = parent_of[c]; d += 1
    return d

depth_of = {c: _depth(c) for c in nodes}
members_of = {}
for c in sorted(nodes, key=lambda c: -depth_of[c]):        # deepest first
    acc = list(own.get(c, ()))
    for kid in children_of[c]:
        acc.extend(members_of[kid])
    members_of[c] = acc

print(f"{len(nodes)} clusters in the tree, depth up to {max(depth_of.values())}, "
      f"root holds {len(members_of[root]):,}")

### The property this corpus exists to exercise

A cluster's children do not exhaust it. The figure below is the share of each internal cluster's
members that belong to **it and to no child of it** — the papers a split leaves behind.

In [ ]:
internal = [c for c in nodes if children_of[c]]
stray = np.array([len(own.get(c, ())) / max(1, len(members_of[c])) for c in internal])

print(f"{len(internal)} clusters have children")
print(f"stray share: mean {stray.mean():.1%}, median {np.median(stray):.1%}, max {stray.max():.1%}")
print(f"clusters whose children exhaust them entirely: {(stray == 0).sum()} of {len(internal)}")
print(f"papers in no selected cluster (HDBSCAN's own noise): {selected_noise:.1%}")

## 5. TF-IDF labels

One label per cluster, in both clusterings: the terms that are far more common in that cluster's
titles than in the corpus at large. A label is a **separate artifact** attached to the cluster it
names, not a field on it — its visibility is its own, because a synthesis can be more sensitive
than any of its sources, and a label that leaks has to be suppressible on its own.

Each label carries the documents it was generated from. That set is what a viewer must be able to
see *entirely* before the label is served to them.

### The scoring, and the obvious version that does not work

**Plain TF-IDF over each cluster treated as one long document gives unusable labels**, and it is
worth recording because it is the first thing anyone writes. Concatenating a cluster's titles makes
every term's frequency enormous, so the ranking is decided almost entirely by inverse document
frequency across the 64-odd cluster "documents" — which rewards whatever is *rarest*. Measured on
this corpus it returned `k_s chocs informatis`, `b_s0 tournament szeg`, `mast 892 ccd`: real tokens,
each unique to one cluster, none of them describing anything.

What works is the same two ingredients weighted the other way round — term frequency **within** the
cluster against document frequency **across the corpus**:

$$\text{score}(t, C) = f(t, C)\,\log\frac{f(t, C)}{f(t, \text{corpus})}$$

where *f* is the share of documents containing the term. A term scores well by being both common in
the cluster and disproportionately so. Two guards do the rest: a term in more than 2% of all titles
is dropped as corpus vocabulary rather than cluster vocabulary, and a term in fewer than 2% of the
cluster's own titles is dropped as a coincidence.

**Label quality tracks the clustering, not this function.** These clusters are drawn in a 2D UMAP
projection, so they are spatially coherent and only roughly topical, and the labels say so. What
this notebook is demonstrating is the *mechanism* — a label is an artifact with its own gate, its
own generating set and its own lifecycle — and that is unaffected by how good the words are.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

# Binary occurrence rather than counts: the score is over the *share of documents* carrying a term,
# so a title repeating a word does not make it more characteristic of the cluster.
MAX_CORPUS_SHARE = 0.02      # above this a term is corpus vocabulary, not cluster vocabulary
MIN_CLUSTER_SHARE = 0.02     # below this it is a coincidence rather than a description

with step("vectorise titles"):
    vec = CountVectorizer(
        stop_words="english",
        # Alphabetic, three characters or more: without this the ranking fills with fragments of
        # identifiers and bare numbers (`b_s0`, `892`), which are distinctive and meaningless.
        token_pattern=r"(?u)\b[a-zA-Z][a-zA-Z-]{2,}\b",
        min_df=10,
        max_df=MAX_CORPUS_SHARE,
        binary=True,
    )
    occurs = vec.fit_transform(title.tolist())
    vocab = np.array(vec.get_feature_names_out())
    corpus_share = np.asarray(occurs.sum(axis=0)).ravel() / occurs.shape[0]

print(f"{len(vocab):,} candidate terms after dropping corpus vocabulary and identifier fragments")


def tfidf_labels(groups, terms=LABEL_TERMS):
    """`{key: [row indices]}` -> `{key: "term term term"}` by the score above."""
    out = {}
    for key, rows in groups.items():
        rows = np.asarray(rows)
        share = np.asarray(occurs[rows].sum(axis=0)).ravel() / max(1, len(rows))
        score = np.where(
            share >= MIN_CLUSTER_SHARE,
            share * np.log(share / (corpus_share + 1e-9) + 1e-9),
            -np.inf,
        )
        top = np.argsort(score)[::-1][:terms]
        chosen = [vocab[t] for t in top if np.isfinite(score[t])]
        out[key] = " ".join(chosen) if chosen else "(no distinctive terms)"
    return out


with step("labels"):
    kmeans_groups = {int(k): np.flatnonzero(kmeans_label == k).tolist() for k in range(KMEANS_K)}
    # Labelling every node of a deep tree means labelling near-duplicates of each other; the
    # clusters worth naming are the ones with enough members to have a distinctive vocabulary.
    hdbscan_groups = {c: members_of[c] for c in nodes if len(members_of[c]) >= mcs}
    kmeans_text  = tfidf_labels(kmeans_groups)
    hdbscan_text = tfidf_labels(hdbscan_groups)

print(f"{len(kmeans_text)} k-means labels, {len(hdbscan_text)} hdbscan labels\n")
for k in list(kmeans_text)[:6]:
    print(f"  kmeans  {k:>4}  {len(kmeans_groups[k]):>7,}  {kmeans_text[k]}")
for c in list(hdbscan_text)[:6]:
    print(f"  hdbscan {c:>6} d{depth_of[c]:<2} {len(members_of[c]):>7,}  {hdbscan_text[c]}")

## 6. Write the build inputs

Four files, in the shapes `tessera build` reads (`annotation-write-cycle.md` §6.1).

**Members are named by source entity id** — the `entity_id` of the points file — and resolved
through the build's own assignment. An id the build did not assign refuses the build rather than
being dropped: a dropped member moves both the count a viewer is shown and the size a proportional
criterion divides by, quietly, in the direction of hiding the cluster.

In [ ]:
KMEANS_LAYER,  KMEANS_LABELS  = "clusters/kmeans", "topics/kmeans"
HDBSCAN_LAYER, HDBSCAN_LABELS = "clusters/hdbscan", "topics/hdbscan"

entity = np.arange(n, dtype=np.uint64)          # dense, 0-based: this corpus's own entity space

with step("write points"):
    # `archive` is the part of a category before the dot (`math`), `primary_category` the whole of
    # the first one (`math.GT`) — the two grains a client filters at.
    primary = np.array([c.split()[0] if c else "unknown" for c in categories])
    archive = np.array([p.split(".")[0] for p in primary])
    pq.write_table(pa.table({
        "entity_id": pa.array(entity, pa.uint64()),
        "x": pa.array(xy[:, 0], pa.float64()),
        "y": pa.array(xy[:, 1], pa.float64()),
        "arxiv_id": pa.array(arxiv_id, pa.string()),
        "archive": pa.array(archive, pa.string()),
        "primary_category": pa.array(primary, pa.string()),
        "submitted_at": pa.array(created.astype("datetime64[us]"), pa.timestamp("us")),
        "title": pa.array(title, pa.string()),
        "abstract": pa.array(abstract, pa.string()),
    }), OUT / "points.parquet")

# --- the quantisation extent, computed rather than assumed -----------------------------------
# **The build's `--extent` must contain the data, and nothing checks that it does.** Coordinates
# are quantised against the extent by `fixed32`, which *clamps*: a point outside collapses onto
# the boundary, the build succeeds, and the map is silently wrong. UMAP output is centred near
# zero and runs negative on both axes, so the grid extent `0,65536,0,65536` — the right answer
# for a corpus whose file already holds Morton codes — would put every negative coordinate on an
# axis and squeeze the rest into a corner a few cells wide.
#
# So the extent is the data's own bounding box with a 2% margin, and it is printed, written to
# the manifest and pasted into the build command below. The margin is not cosmetic: the extent
# is a half-open interval, and a point exactly at the maximum quantises to the clamp.
_pad = 0.02 * max(np.ptp(xy[:, 0]), np.ptp(xy[:, 1]))
EXTENT = (float(xy[:, 0].min() - _pad), float(xy[:, 0].max() + _pad),
          float(xy[:, 1].min() - _pad), float(xy[:, 1].max() + _pad))
EXTENT_ARG = ",".join(f"{v:.4f}" for v in EXTENT)
print(f"extent   {EXTENT_ARG}")

with step("write pairs"):
    # **The access relation, and therefore the access control.** One term per category a paper
    # carries, so a principal granted `math.GT` sees exactly the papers filed under it. Term ids are
    # dense and assigned by descriptor sort, which is the corpus's existing convention.
    per_paper = [c.split() if c else [] for c in categories]
    vocab = sorted({t for cats in per_paper for t in cats})
    code_of = {t: i for i, t in enumerate(vocab)}
    ent_col, term_col = [], []
    for i, cats in enumerate(per_paper):
        for t in cats:
            ent_col.append(i); term_col.append(code_of[t])
    order = np.lexsort((np.array(ent_col), np.array(term_col)))   # (term, entity), the build's order
    pq.write_table(pa.table({
        "entity_id": pa.array(np.array(ent_col, dtype=np.uint32)[order], pa.uint32()),
        "term_id": pa.array(np.array(term_col, dtype=np.uint32)[order], pa.uint32()),
    }), OUT / "pairs.parquet")
    pq.write_table(pa.table({
        "term_id": pa.array(np.arange(len(vocab), dtype=np.uint32), pa.uint32()),
        "descriptor": pa.array(vocab, pa.string()),
    }), OUT / "terms.parquet")

print(f"{len(ent_col):,} pairs over {len(vocab)} terms")

### The artifacts and their members

A cluster is one row with no content. A label is one row per variation, carrying the text and
naming the cluster it attaches to.

**The HDBSCAN clusters carry `parent_key`**, and that is the whole hierarchy: an edge relates two
artifacts of one level, the parent direction is what is stored, and the child direction is the same
relation read the other way. K-means clusters carry none — they are peers.

In [ ]:
def cluster_key(prefix, k):
    return f"{prefix}-{k:06d}"

with step("write artifacts"):
    a_layer, a_key, a_var, a_vals = [], [], [], []
    a_att_layer, a_att_key, a_parent = [], [], []

    def artifact(layer, key, *, variation=None, values=None,
                 attached=None, parent=None):
        a_layer.append(layer); a_key.append(key)
        a_var.append(variation); a_vals.append(values)
        a_att_layer.append(attached[0] if attached else None)
        a_att_key.append(attached[1] if attached else None)
        a_parent.append(parent)

    for k in range(KMEANS_K):
        artifact(KMEANS_LAYER, cluster_key("km", k))
    for c in nodes:
        parent = parent_of.get(c)
        artifact(HDBSCAN_LAYER, cluster_key("hdb", c),
                 parent=cluster_key("hdb", parent) if parent is not None else None)

    # Labels, two ranked variations each: the specific description first, a generic fallback
    # second. A viewer is served the first whose generating set they hold entirely, or nothing —
    # never the cluster's identity with its description missing.
    for k, text in kmeans_text.items():
        for v, val in enumerate([text, "a cluster of papers"]):
            artifact(KMEANS_LABELS, f"kml-{k:06d}", variation=v, values=[val],
                     attached=(KMEANS_LAYER, cluster_key("km", k)))
    for c, text in hdbscan_text.items():
        for v, val in enumerate([text, "a cluster of papers"]):
            artifact(HDBSCAN_LABELS, f"hdbl-{c:06d}", variation=v, values=[val],
                     attached=(HDBSCAN_LAYER, cluster_key("hdb", c)))

    pq.write_table(pa.table({
        "layer": pa.array(a_layer, pa.string()),
        "stable_key": pa.array(a_key, pa.string()),
        "variation": pa.array(a_var, pa.uint32()),
        "values": pa.array(a_vals, pa.list_(pa.string())),
        "attached_layer": pa.array(a_att_layer, pa.string()),
        "attached_key": pa.array(a_att_key, pa.string()),
        "parent_key": pa.array(a_parent, pa.string()),
    }), OUT / "artifacts.parquet")

with step("write members"):
    m_layer, m_key, m_var, m_member = [], [], [], []

    def members(layer, key, rows, variation=None):
        for r in rows:
            m_layer.append(layer); m_key.append(key)
            m_var.append(variation); m_member.append(int(r))

    for k in range(KMEANS_K):
        members(KMEANS_LAYER, cluster_key("km", k), kmeans_groups[k])
    for c in nodes:
        members(HDBSCAN_LAYER, cluster_key("hdb", c), members_of[c])

    # A label's own membership is the documents it was generated from, and each variation's
    # generating set is drawn from that same membership — a generating set naming a non-member is
    # a different object, and the model refuses it.
    lrng = np.random.default_rng(SEED)
    def label_members(layer, key, pool):
        pool = np.asarray(pool)
        pick = pool if len(pool) <= LABEL_SAMPLE else lrng.choice(pool, LABEL_SAMPLE, replace=False)
        members(layer, key, pick)                       # the label's membership
        members(layer, key, pick, variation=0)          # the specific variation's sources
        # The fallback variation is generated from a third of them, so it is satisfiable by a
        # narrower principal than the specific one — which is the point of ranking them.
        members(layer, key, pick[: max(1, len(pick) // 3)], variation=1)

    for k in kmeans_text:
        label_members(KMEANS_LABELS, f"kml-{k:06d}", kmeans_groups[k])
    for c in hdbscan_text:
        label_members(HDBSCAN_LABELS, f"hdbl-{c:06d}", members_of[c])

    pq.write_table(pa.table({
        "layer": pa.array(m_layer, pa.string()),
        "stable_key": pa.array(m_key, pa.string()),
        "variation": pa.array(m_var, pa.uint32()),
        "member": pa.array(np.array(m_member, dtype=np.uint64), pa.uint64()),
    }), OUT / "members.parquet")

print(f"{len(a_key):,} artifact rows, {len(m_key):,} member rows")

## 7. The configuration that reads it

`schema.toml` declares the attribute columns and where each lives; `layers.toml` declares the four
layers. Two properties in `layers.toml` are disclosure controls and therefore have **no default**,
which is why every layer states them explicitly:

- **the gate** — whether a viewer may know the layer exists at all. The defaultable value would be
  *no gate*, which is the widest one there is, so the file says `gate` or `ungated = true`.
- **`visible_when`** — the masked count or share a cluster must clear to be served. Absent, it
  would serve the existence and count of every cluster down to a single member.

The HDBSCAN layer takes the **proportional** form and the k-means layer the absolute one, so the
corpus exercises both. They behave differently in a way worth knowing about: a share does not
shrink as you descend, so under the proportional form a small child can clear the bar while its
large parent fails it. The lineage then has holes — each cluster having passed its own test — and a
layer declaring that form must expect them.

In [ ]:
SCHEMA = """
# Generated by notebooks/arxiv-corpus.ipynb. Edit the notebook, not this.

[[attribute]]
name       = "archive"
type       = "category"
width      = "u8"
render     = true
index      = true
vocabulary = "declared"
values_key = "archive"
# arXiv's archive names are published taxonomy, so their existence discloses nothing about this
# corpus's contents. `public` additionally requires `declared`, which it is.
listing    = "public"

[[attribute]]
name       = "primary_category"
type       = "category"
width      = "u16"
render     = true
index      = true
vocabulary = "declared"
values_key = "primary_category"
listing    = "public"

# `timestamp_us` rather than `i64`, so the unit is a fact the client reads off `/v1/meta` rather
# than a convention it has to be told.
[[attribute]]
name   = "submitted_at"
type   = "timestamp_us"
render = true

# Prose lives in the record blob and reaches a client at drill-down: `render` on a text column is
# refused, the hot column being a fixed-width slot per row. `index` builds the token index behind
# `match` and `phrase`.
[[attribute]]
name   = "title"
type   = "text"
index  = true

[[attribute]]
name   = "abstract"
type   = "text"
index  = true

[[attribute]]
name   = "arxiv_id"
type   = "keyword"
index  = true
"""

LAYERS = f"""
# Generated by notebooks/arxiv-corpus.ipynb. Edit the notebook, not this.
#
# Four layers: two clusterings and a label set over each. The clusterings are deliberately
# different shapes — k-means flat, HDBSCAN a tree whose children do not exhaust their parents.

[[layer]]
name = "{KMEANS_LAYER}"
title = "k-means clusters"
slices = ["s0"]
membership = "enumerated"
ungated = true
artifacts_carry_own = false
# The absolute form: a fixed floor, whatever the cluster's size.
visible_when = {{ min_visible = 50 }}
hierarchy = {{ kind = "flat", prune_children = false }}
content = {{ derived = ["centroid", "box"] }}

[[layer]]
name = "{HDBSCAN_LAYER}"
title = "HDBSCAN clusters"
slices = ["s0"]
membership = "enumerated"
ungated = true
artifacts_carry_own = false
# The proportional form: a share of the cluster's own size, so the bar scales. Expect holes in the
# lineage — a share does not shrink downward, so a small child can clear what its parent cannot.
visible_when = {{ min_fraction = 0.05 }}
hierarchy = {{ kind = "nested", prune_children = false }}
content = {{ derived = ["centroid", "box"] }}

[[layer]]
name = "{KMEANS_LABELS}"
title = "k-means topics"
slices = ["s0"]
membership = "enumerated"
ungated = true
artifacts_carry_own = false
visible_when = "none"
hierarchy = {{ kind = "flat" }}
depends_on = ["{KMEANS_LAYER}"]

[[layer.content.supplied]]
kind = "label_text"
corpus_derived = true

[[layer]]
name = "{HDBSCAN_LABELS}"
title = "HDBSCAN topics"
slices = ["s0"]
membership = "enumerated"
ungated = true
artifacts_carry_own = false
visible_when = "none"
hierarchy = {{ kind = "flat" }}
depends_on = ["{HDBSCAN_LAYER}"]

[[layer.content.supplied]]
kind = "label_text"
corpus_derived = true
"""

with step("write config"):
    (OUT / "schema.toml").write_text(SCHEMA.lstrip())
    (OUT / "layers.toml").write_text(LAYERS.lstrip())

    # The two category vocabularies, in the shape `--values` binds.
    for name, values in [("archive", sorted(set(archive))),
                         ("primary_category", sorted(set(primary)))]:
        pq.write_table(pa.table({
            "key": pa.array(values, pa.string()),
            # Code 0 is reserved for *absent*, so declared codes start at one.
            "code": pa.array(np.arange(1, len(values) + 1, dtype=np.uint32), pa.uint32()),
            "label": pa.array(values, pa.string()),
        }), OUT / f"{name}.parquet")

print((OUT / "layers.toml").read_text()[:400])

## 8. What this run produced

The manifest records the parameters and the shape of the output, so a bundle can be traced back to
the run that made it — and so two runs can be compared without re-reading the data.

In [ ]:
manifest = {
    "sample": n,
    "corpus": n_full,
    "seed": SEED,
    "umap": "recomputed" if RECOMPUTE_UMAP else "reused from geometry.parquet",
    "umap_params": UMAP_PARAMS if RECOMPUTE_UMAP else None,
    # The build's `--extent`, computed from the geometry rather than assumed. A bundle built
    # against any other extent holds different quantised positions for the same points.
    "extent": EXTENT_ARG,
    "kmeans": {"k": KMEANS_K, "labels": len(kmeans_text)},
    "hdbscan": {
        "min_cluster_size": int(mcs),
        "clusters_in_tree": len(nodes),
        "selected_clusters": int(len(set(clusterer.labels_)) - 1),
        "max_depth": int(max(depth_of.values())),
        "noise_share": round(selected_noise, 4),
        "stray_share_mean": round(float(stray.mean()), 4),
        "labels": len(hdbscan_text),
    },
    "terms": len(vocab),
    "pairs": len(ent_col),
    "artifact_rows": len(a_key),
    "member_rows": len(m_key),
    "seconds": STEPS,
    "total_seconds": round(time.time() - T0, 1),
}
(OUT / "manifest.json").write_text(json.dumps(manifest, indent=2))
print(json.dumps(manifest, indent=2))

### Checks

These are the properties the rest of the system depends on, asserted here rather than assumed —
each one is a thing that fails silently downstream if it is wrong.

In [ ]:
# Every member names a paper this corpus holds. A member outside the range refuses the build, but
# it is worth failing here, where the line that produced it is in view.
assert min(m_member) >= 0 and max(m_member) < n, "a member names an entity outside the corpus"

# A child's members are a subset of its parent's. This is what makes rollup sound under an absolute
# criterion — a child's masked count can never exceed its parent's — and the build reports every
# edge that breaks it. The condensed tree cannot produce a violation; the assertion is here because
# the *member computation above* could.
for c, p in parent_of.items():
    if not set(members_of[c]) <= set(members_of[p]):
        raise AssertionError(f"cluster {c} escapes its parent {p}")

# And they do not exhaust it, which is the property a planted tree would not have.
assert (stray > 0).sum() > 0, "no cluster keeps members its children do not — the tree is covering"

# Every label attaches to a cluster that exists.
keys = set(zip(a_layer, a_key))
for layer, att_layer, att_key in zip(a_layer, a_att_layer, a_att_key):
    if att_key is not None:
        assert (att_layer, att_key) in keys, f"{layer} attaches to a missing {att_layer}/{att_key}"

# Every generating set is a subset of its artifact's own membership.
by_artifact = {}
for layer, key, var, mem in zip(m_layer, m_key, m_var, m_member):
    by_artifact.setdefault((layer, key, var), set()).add(mem)
for (layer, key, var), rows in by_artifact.items():
    if var is None:
        continue
    assert rows <= by_artifact[(layer, key, None)], \
        f"{layer}/{key} variation {var} names a document it does not hold"

print("all checks passed")
print(f"\nwrote to {OUT}:")
for f in sorted(OUT.iterdir()):
    print(f"  {f.name:28} {f.stat().st_size / 1e6:8.2f} MB")

## 9. Building a bundle from it

From the repository root, with the binary built (`cargo build --release -p tessera-cli`). The
extent comes from `manifest.json`, which is why it is read back rather than typed:

```bash
OUT=${TESSERA_NOTEBOOK_OUT:-$TESSERA_DATA/notebook}
EXTENT=$(python3 -c "import json,sys; print(json.load(open('$OUT/manifest.json'))['extent'])")

./target/release/tessera build \
  --points  "$OUT/points.parquet" \
  --pairs   "$OUT/pairs.parquet" \
  --schema  "$OUT/schema.toml" \
  --layers  "$OUT/layers.toml" \
  --artifacts "$OUT/artifacts.parquet" \
  --artifact-members "$OUT/members.parquet" \
  --values  "archive=$OUT/archive.parquet" \
  --values  "primary_category=$OUT/primary_category.parquet" \
  --out     bundles/notebook \
  --slice s0 --extent "$EXTENT" --mint-id-key
```

**The extent must contain the data, and the build does not check that it does.** Coordinates are
quantised against it by clamping, so a point outside lands on the boundary and the build reports
nothing: the bundle is well-formed and the map is wrong. The grid extent `0,65536,0,65536` is
right only for a file that already holds Morton codes; this notebook writes real coordinates, so
it computes the bounding box (§6) and passes that.

The build writes `reports/containment.json` beside the bundle: every parent/child edge whose child
holds a member its parent does not. It should be empty for this corpus — the checks above assert
the same property — and an entry in it means the member computation and the tree have diverged.

A grant is written in term descriptors, which `terms.parquet` maps back to category names. A
principal holding a single narrow category is the interesting one: they see a few clusters, each
with a masked count that is their own, and the labels only where they hold the whole generating set.